<a href="https://colab.research.google.com/github/lailaaagha/pork-rasher-defect-detection/blob/main/%20%20%20notebooks/MMA3001_Meat_Defect_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi


Thu Oct 22 06:51:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   58C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q ultralytics roboflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.0/47.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 373.9/373.9 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 114.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.9/97.9 kB 10.0 MB/s eta 0:00:00


In [3]:
from google.colab import userdata
from roboflow import Roboflow
import yaml

# Download dataset v2 (70/20/10 split) in YOLOv8 format.
# The API key is read from Colab Secrets so it is never stored in the notebook.
rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
project = rf.workspace("lailaa-agha").project("pork-rasher-error-packaging-jybag")
project.version(2).download("yolov8", location="/content/data")

# Fix the paths in data.yaml (Roboflow writes '../train/images', which breaks).
yaml_path = "/content/data/data.yaml"
cfg = yaml.safe_load(open(yaml_path))
cfg.update(path="/content/data", train="train/images",
           val="valid/images", test="test/images")
yaml.safe_dump(cfg, open(yaml_path, "w"))
print("Classes:", cfg["names"])

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /content/data in yolov8:: 100%|██████████| 3914/3914 [00:03<00:00, 1144.47it/s]


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Classes: ['loose-meat', 'packaging-error', 'twisted-meat', 'unsealed', 'wrinkle']


In [4]:
!ls /content/data/train/images | wc -l
!ls /content/data/valid/images | wc -l
!ls /content/data/test/images | wc -l

1369
391
195


In [5]:
from pathlib import Path

# Convert any polygon labels (class x1 y1 x2 y2 ...) to their enclosing box
# (class x_c y_c w h), the same fix used in src/rasher/explore.py.
fixed = 0
for lbl in Path("/content/data").glob("*/labels/*.txt"):
    out, changed = [], False
    for line in lbl.read_text().splitlines():
        p = line.split()
        if len(p) > 5:  # polygon line
            xs = [float(v) for v in p[1::2]]
            ys = [float(v) for v in p[2::2]]
            line = (f"{p[0]} {(min(xs)+max(xs))/2:.6f} {(min(ys)+max(ys))/2:.6f} "
                    f"{max(xs)-min(xs):.6f} {max(ys)-min(ys):.6f}")
            changed = True
            fixed += 1
        out.append(line)
    if changed:
        lbl.write_text("\n".join(out) + "\n")
        print("Fixed:", lbl.name)

# Delete YOLO's label caches so it re-reads the corrected files.
for cache in Path("/content/data").glob("*/labels.cache"):
    cache.unlink()
print(f"Converted {fixed} polygon line(s) to boxes.")

Fixed: frame_with_red_9524_jpg.rf.f93b9784e87d9b449db728f963f07efa.txt
Fixed: frame_with_red_10514_jpg.rf.c9769dc5edf123991c2d316edd0c4596.txt
Converted 2 polygon line(s) to boxes.


In [6]:
from google.colab import drive
drive.mount('/content/drive')

# All training results are saved here so they survive a Colab disconnect.
RUNS = "/content/drive/MyDrive/MMA3001_runs"

Mounted at /content/drive


In [9]:
from ultralytics import YOLO

# Start from YOLOv8n weights pre-trained on the COCO dataset (transfer learning).
model = YOLO("yolov8n.pt")

model.train(
    data="/content/data/data.yaml",
    epochs=50,          # max passes through the training set
    patience=15,        # stop early if validation mAP doesn't improve for 15 epochs
    imgsz=640,          # images resized to 640 px (longest side) before entering the network
    batch=16,           # images processed together per step
    seed=0,             # fixed random seed so the run is repeatable
    deterministic=True,
    project=RUNS,
    name="yolov8n_640",
    exist_ok=True,
)


Ultralytics 8.4.183 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_640, nbs=64, nms=None, opset=None, optimize=False, optimize

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7e6bd2102ac0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
        

In [10]:
from ultralytics import YOLO

# Alternative model: YOLOv8s ("small") — ~3.7x more parameters than YOLOv8n.
# Same settings as YOLOv8n so the comparison is fair (only the model changes).
model_s = YOLO("yolov8s.pt")
model_s.train(
    data="/content/data/data.yaml",
    epochs=50, patience=15, imgsz=640, batch=16,
    seed=0, deterministic=True,
    project=RUNS, name="yolov8s_640", exist_ok=True,
)

Ultralytics 8.4.184 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8s_640, nbs=64, nms=None, opset=None, optimize=False, optimize

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7dbf8c4d8050>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
        

In [8]:
# Sensitivity analysis: input resolution.
# Retrain YOLOv8n with identical settings, changing ONLY imgsz.
# We already have 640 px (yolov8n_640). Lower resolutions are faster but
# shrink small defects (packaging-error is only ~25 px wide at 640).
from ultralytics import YOLO

# Sensitivity analysis: input resolution.
...
for size in [320, 480]:
    m = YOLO("yolov8n.pt")
    m.train(
        data="/content/data/data.yaml",
        epochs=50, patience=15, imgsz=size, batch=16,
        seed=0, deterministic=True,
        project=RUNS, name=f"yolov8n_{size}", exist_ok=True,
    )

Ultralytics 8.4.184 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_320, nbs=64, nms=None, opset=None, optimize=False, optimize

In [9]:
import os
import numpy as np
import pandas as pd
from ultralytics import YOLO

# Folder in Google Drive for all analysis outputs.
OUT = f"{RUNS}/analysis"
os.makedirs(OUT, exist_ok=True)

# Every trained model and the image size it was trained at.
RUNS_TO_EVAL = {
    "yolov8n_320": 320,
    "yolov8n_480": 480,
    "yolov8n_640": 640,
    "yolov8s_640": 640,
}

rows = []
for name, size in RUNS_TO_EVAL.items():
    model = YOLO(f"{RUNS}/{name}/weights/best.pt")
    m = model.val(data="/content/data/data.yaml", split="val", imgsz=size,
                  batch=16, half=False, plots=True, verbose=False,
                  project=OUT, name=f"val_{name}", exist_ok=True)

    # One row per model: overall scores + AP50 for each defect class.
    row = {"run": name, "imgsz": size, "mAP50": m.box.map50, "mAP50-95": m.box.map,
           "precision": m.box.mp, "recall": m.box.mr}
    for ci, ap in zip(m.box.ap_class_index, m.box.ap50):
        row[f"AP50_{m.names[ci]}"] = ap
    rows.append(row)

    # Save the curves behind the scores, for the numerical-methods analysis on the laptop:
    #   pr_precision: precision vs recall (AP = area under this curve)
    #   p_curve / r_curve: precision and recall vs confidence threshold
    curves = m.box.curves_results
    np.savez(f"{OUT}/curves_{name}.npz",
             recall_grid=curves[0][0], pr_precision=curves[0][1],
             conf_grid=curves[2][0], p_curve=curves[2][1], r_curve=curves[3][1],
             ap50=m.box.ap50, class_index=m.box.ap_class_index,
             n_gt=m.nt_per_class,
             names=np.array([m.names[i] for i in range(len(m.names))]))

summary = pd.DataFrame(rows).round(3)
summary.to_csv(f"{OUT}/validation_summary.csv", index=False)
summary

WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
Ultralytics 8.4.184 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,133,679 parameters, 0 gradients, 8.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 4996.5±645.2 MB/s, size: 235.2 KB)
val: Scanning /content/data/valid/labels.cache... 391 images, 3 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 391/391 136.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 25/25 3.9it/s 6.4s
                   all        391        486      0.645      0.568      0.528      0.288
Speed: 0.8ms preprocess, 2.6ms inference, 0.0ms loss, 2.9ms postprocess per image
Results saved to /content/drive/MyDrive/MMA3001_runs/analysis/val_yolov8n_320
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
Ultralytics 8.4.184 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 

,run,imgsz,mAP50,mAP50-95,precision,recall,AP50_loose-meat,AP50_packaging-error,AP50_twisted-meat,AP50_unsealed,AP50_wrinkle
0,yolov8n_320,320,0.528,0.288,0.645,0.568,0.750,0.310,0.736,0.846,0.000
1,yolov8n_480,480,0.584,0.298,0.696,0.563,0.942,0.381,0.748,0.849,0.000
2,yolov8n_640,640,0.561,0.288,0.675,0.577,0.924,0.370,0.673,0.836,0.001
3,yolov8s_640,640,0.543,0.289,0.739,0.519,0.858,0.355,0.642,0.860,0.000


In [10]:
import time
import copy
import glob
import cv2
import torch
from ultralytics.utils.torch_utils import get_flops

def load_net(name):
    """Load a trained network, fuse Conv+BatchNorm layers (as in deployment), move to GPU."""
    return YOLO(f"{RUNS}/{name}/weights/best.pt").model.fuse(verbose=False).cuda().eval()

def first_output(out):
    """The network returns (predictions, features); keep the predictions tensor."""
    return out[0] if isinstance(out, (list, tuple)) else out

def time_net(net, size, half, reps=100, warmup=20):
    """Median batch-1 GPU inference time in ms (synchronised so we time the GPU work)."""
    x = torch.rand(1, 3, size, size, device="cuda")
    x = x.half() if half else x
    times = []
    with torch.no_grad():
        for i in range(warmup + reps):
            torch.cuda.synchronize(); t0 = time.perf_counter()
            net(x)
            torch.cuda.synchronize(); t1 = time.perf_counter()
            if i >= warmup:
                times.append((t1 - t0) * 1000)
    return float(np.median(times)), float(np.std(times))

# 1) Speed and FLOPs for every model, in FP32 and FP16
rows = []
for name, size in RUNS_TO_EVAL.items():
    for half in (False, True):
        net = load_net(name)
        gflops = get_flops(net, size)          # GFLOPs per image at this input size
        if half:
            net = net.half()
        ms, sd = time_net(net, size, half)
        rows.append({"run": name, "imgsz": size, "precision": "FP16" if half else "FP32",
                     "GFLOPs": round(gflops, 2), "latency_ms": round(ms, 2), "std_ms": round(sd, 2),
                     "achieved_TFLOP_per_s": round(gflops / ms, 2)})   # GFLOP per ms = TFLOP/s
speed = pd.DataFrame(rows)
speed.to_csv(f"{OUT}/speed_flops.csv", index=False)
print(speed.to_string(index=False))

# 2) Numerical (round-off) error: same image through FP32 and FP16 copies of YOLOv8n-640
net32 = load_net("yolov8n_640")
net16 = copy.deepcopy(net32).half()
errs = []
for path in sorted(glob.glob("/content/data/valid/images/*.jpg"))[:50]:
    img = cv2.resize(cv2.imread(path), (640, 640))[:, :, ::-1].copy()   # BGR -> RGB
    x = torch.from_numpy(img).permute(2, 0, 1)[None].float().div(255).cuda()
    with torch.no_grad():
        y32 = first_output(net32(x))
        y16 = first_output(net16(x.half())).float()
    # Output rows 0-3 = box coordinates (pixels), rows 4+ = class scores (0-1)
    errs.append({"box_max_abs_err_px": (y32[:, :4] - y16[:, :4]).abs().max().item(),
                 "score_max_abs_err": (y32[:, 4:] - y16[:, 4:]).abs().max().item()})
errs = pd.DataFrame(errs)
errs.to_csv(f"{OUT}/fp16_roundoff.csv", index=False)
print("\nFP16 vs FP32 round-off error over 50 validation images:")
print(errs.describe().loc[["mean", "max"]])

# 3) Does FP16 change accuracy?
m16 = YOLO(f"{RUNS}/yolov8n_640/weights/best.pt").val(
    data="/content/data/data.yaml", split="val", imgsz=640, batch=16,
    half=True, plots=False, verbose=False, project=OUT, name="val_fp16", exist_ok=True)
print(f"\nYOLOv8n-640 mAP50  FP32: {summary.loc[summary.run == 'yolov8n_640', 'mAP50'].item():.4f}"
      f"   FP16: {m16.box.map50:.4f}")

        run  imgsz precision  GFLOPs  latency_ms  std_ms  achieved_TFLOP_per_s
yolov8n_320    320      FP32    2.16        5.65    0.90                  0.38
yolov8n_320    320      FP16    2.16        6.82    0.92                  0.32
yolov8n_480    480      FP32    4.86        5.72    0.84                  0.85
yolov8n_480    480      FP16    4.86        6.95    1.08                  0.70
yolov8n_640    640      FP32    8.64        7.09    1.52                  1.22
yolov8n_640    640      FP16    8.64        6.79    1.08                  1.27
yolov8s_640    640      FP32   28.44       17.14    0.18                  1.66
yolov8s_640    640      FP16   28.44        6.84    1.17                  4.16

FP16 vs FP32 round-off error over 50 validation images:
      box_max_abs_err_px  score_max_abs_err
mean            1.252354           0.000837
max             2.631989           0.002435
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
Ultralyti

In [11]:
from pathlib import Path

# ===== FINAL EVALUATION ON THE TEST SET (run once) =====
# Model and threshold were chosen using the validation set only.
FINAL_RUN, FINAL_SIZE, T_OPT = "yolov8n_480", 480, 0.137
final = YOLO(f"{RUNS}/{FINAL_RUN}/weights/best.pt")

# 1) Standard detection metrics on the test split (prints a per-class table)
mt = final.val(data="/content/data/data.yaml", split="test", imgsz=FINAL_SIZE,
               batch=16, half=False, plots=True, verbose=True,
               project=OUT, name="test_yolov8n_480", exist_ok=True)
curves = mt.box.curves_results
np.savez(f"{OUT}/curves_test_{FINAL_RUN}.npz",
         recall_grid=curves[0][0], pr_precision=curves[0][1],
         conf_grid=curves[2][0], p_curve=curves[2][1], r_curve=curves[3][1],
         ap50=mt.box.ap50, class_index=mt.box.ap_class_index, n_gt=mt.nt_per_class,
         names=np.array([mt.names[i] for i in range(len(mt.names))]))

# 2) Tray-level decisions: for each test image, which defects are really there
#    vs which the model flags, at the default (0.25) and optimised threshold.
rows = []
img_paths = sorted(str(p) for p in Path("/content/data/test/images").glob("*.jpg"))
for res in final.predict(source=img_paths, imgsz=FINAL_SIZE, conf=0.01, verbose=False, stream=True):
    p = Path(res.path)
    lbl = Path("/content/data/test/labels") / f"{p.stem}.txt"
    gt = sorted({int(l.split()[0]) for l in lbl.read_text().splitlines() if l.strip()}) if lbl.exists() else []
    conf = res.boxes.conf.cpu().numpy()
    cls = res.boxes.cls.cpu().numpy().astype(int)
    for t in (0.25, T_OPT):
        pred = sorted(set(cls[conf >= t].tolist()))
        rows.append({"image": p.name, "threshold": t, "gt_classes": gt, "pred_classes": pred,
                     "actually_defective": len(gt) > 0, "flagged_defective": len(pred) > 0})
trays = pd.DataFrame(rows)
trays.to_csv(f"{OUT}/test_tray_decisions.csv", index=False)

for t, g in trays.groupby("threshold"):
    print(f"\n===== Threshold {t} =====")
    print(pd.crosstab(g.actually_defective, g.flagged_defective,
                      rownames=["actually defective"], colnames=["flagged defective"]))
    print("Per defect type: share of test trays containing it where the model flagged it")
    for ci, nm in final.names.items():
        has = g[g.gt_classes.apply(lambda c: ci in c)]
        if len(has):
            found = has.pred_classes.apply(lambda c: ci in c).mean()
            print(f"  {nm:<16} trays={len(has):>3}  flagged={found:.1%}")

# 3) Save 12 example test predictions (at the optimised threshold) for the report
final.predict(source=img_paths[:12], imgsz=FINAL_SIZE, conf=T_OPT, save=True,
              project=OUT, name="test_examples", exist_ok=True, verbose=False)
print(f"\nSaved example images to {OUT}/test_examples")

WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
Ultralytics 8.4.184 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,133,679 parameters, 0 gradients, 8.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3830.8±693.5 MB/s, size: 238.7 KB)
val: Scanning /content/data/test/labels... 195 images, 3 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 195/195 1.6Kit/s 0.1s
val: New cache created: /content/data/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 13/13 3.7it/s 3.6s
                   all        195        243      0.629      0.473      0.461      0.201
            loose-meat         10         11      0.674      0.818      0.763      0.309
       packaging-error         89        121      0.358       0.19      0.252     0.0678
          twisted-meat         18         19      0.394      0.579       0.49      0.206
